In [2]:
import numpy as np
import torch
from torch_geometric.data import Data
from tqdm import tqdm
import pandas as pd

In [3]:

def get_xy(location):
    """
    location: lista [x, y] albo NaN/None
    """
    if isinstance(location, (list, tuple, np.ndarray)) and len(location) >= 2:
        return float(location[0]), float(location[1])

    return np.nan, np.nan


In [4]:

def build_near_player_distance_edges(position: torch.tensor, radius: float = 15):
    edges = []
    edge_attrs = []

    num_nodes = position.size(0)

    for i in range(num_nodes):
        for j in range(num_nodes):
            if i ==j:
                continue
            dx = position[j,0] - position[i,0]
            dy = position[j,1] - position[i,1]

            distance = torch.sqrt(dx ** 2 + dy ** 2)

            if distance <= radius:
                edges.append([i,j])
                edge_attrs.append([dx.item()/120, dy.item()/80, distance.item()/120])
    if len(edges) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 3), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    return edge_index, edge_attr


In [5]:

def add_actor_edges(edge_index, edge_attr, positions, actor_idx):
    """
    Dodaje krawędzie actor -> wszyscy oraz wszyscy -> actor.
    """
    existing_edges = set()

    for k in range(edge_index.size(1)):
        i = int(edge_index[0, k])
        j = int(edge_index[1, k])
        existing_edges.add((i, j))

    new_edges = []
    new_attrs = []

    num_nodes = positions.size(0)

    for j in range(num_nodes):
        if j == actor_idx:
            continue

        for i, target in [(actor_idx, j), (j, actor_idx)]:
            if (i, target) in existing_edges:
                continue

            dx = positions[target, 0] - positions[i, 0]
            dy = positions[target, 1] - positions[i, 1]
            distance = torch.sqrt(dx ** 2 + dy ** 2)

            new_edges.append([i, target])
            new_attrs.append([
                dx.item() / 120.0,
                dy.item() / 80.0,
                distance.item() / 120.0,
            ])

    if len(new_edges) == 0:
        return edge_index, edge_attr

    new_edge_index = torch.tensor(new_edges, dtype=torch.long).t().contiguous()
    new_edge_attr = torch.tensor(new_attrs, dtype=torch.float)

    edge_index = torch.cat([edge_index, new_edge_index], dim=1)
    edge_attr = torch.cat([edge_attr, new_edge_attr], dim=0)

    return edge_index, edge_attr




In [6]:

def build_graph_for_event(event_row: pd.DataFrame, frame_row: pd.DataFrame, radius = 15.0):

    if len(frame_row) == 0:
        return None
    
    #event location
    event_x, event_y = get_xy(event_row["location"])

    if np.isnan(event_x) or np.isnan(event_y):
        return None
    
    #players postions
    players_postions = []
    for _, row in frame_row.iterrows():
        x, y = get_xy(row["location"])

        if np.isnan(x) or np.isnan(y):
            continue
        players_postions.append([x,y])

    if len(players_postions) < 2:
        return None
    
    players_postions = torch.tensor(players_postions, dtype = torch.float)

    #node features
    node_features = []
    actor_mask = []

    for _,row in frame_row.iterrows():
        x,y = get_xy(row["location"])

        if np.isnan(x) or np.isnan(y):
            continue
        
        is_teammate = float(bool(row.get("teammate", False)))
        is_opponent = 1.0 - is_teammate
        is_actor = float(bool(row.get("actor", False)))
        is_keeper = float(bool(row.get("keeper", False)))

        dx_event = x - event_x
        dy_event = y - event_y
        distance_to_ball = np.sqrt(dx_event ** 2 + dy_event ** 2)

        distance_to_own_goal = np.sqrt((x - 0.0) ** 2 + (y - 40.0) ** 2)
        distance_to_opponent_goal = np.sqrt((x - 120.0) ** 2 + (y - 40.0) ** 2)

        features = [
            x/120.0,
            y/80.0,
            is_teammate,
            is_opponent,
            is_actor,
            is_keeper,
            distance_to_ball/120.0,
            distance_to_own_goal/120.0,
            distance_to_opponent_goal/120.0,
        ]
        node_features.append(features)
        actor_mask.append(bool(is_actor))


    x_tensor = torch.tensor(node_features, dtype=torch.float)
    actor_mask = torch.tensor(actor_mask, dtype=torch.bool)

    if actor_mask.sum().item() != 1:
        return None
    
    actor_idx = int(torch.where(actor_mask)[0][0])

    #edges creating
    edge_index, edge_attr = build_near_player_distance_edges(position=players_postions, radius = radius)
    edge_index, edge_attr = add_actor_edges(
        edge_index=edge_index,
        edge_attr=edge_attr,
        positions=players_postions,
        actor_idx=actor_idx,
    )
    #global event features
    event_type = event_row["type"]
    event_type_pass = float(event_type == "Pass")
    event_type_carry = float(event_type == "Carry")
    event_type_dribble = float(event_type == "Dribble")
    event_type_receipt = float(event_type == "Ball Receipt*")

    minute = float(event_row.get("minute", 0.0))
    second = float(event_row.get("second", 0.0))

    event_features = torch.tensor([
        event_x / 120.0,
        event_y / 80.0,
        minute / 120.0,
        second / 60.0,
        event_type_pass,
        event_type_carry,
        event_type_dribble,
        event_type_receipt,
    ], dtype=torch.float).unsqueeze(0)

    y_pressure = torch.tensor([float(event_row["y_pressure"])], dtype = torch.float)
    y_turnover = torch.tensor([float(event_row["y_turnover"])], dtype = torch.float)

    graph = Data(
        x=x_tensor,
        edge_index=edge_index,
        edge_attr=edge_attr,
        actor_mask = actor_mask,
        event_features = event_features,
        y_pressure = y_pressure,
        y_turnover = y_turnover,
    )

    graph.event_id = event_row["id"]
    graph.match_id = int(event_row["match_id"])
    graph.event_index = int(event_row["index"])

    return graph



        




In [ ]:

def build_graph_dataset(events_df: pd.DataFrame, frames_df: pd.DataFrame, radius = 15.0):
    graphs = []

    frames_by_event = {
        event_id: group
        for event_id, group in frames_df.groupby("id")
    }

    for _,event_row in tqdm(events_df.iterrows(), total=len(events_df)):
        event_id = event_row["id"]

        frame_rows = frames_by_event.get(event_id)

        if frame_rows is None:
            continue

        graph = build_graph_for_event(
            event_row=event_row,
            frame_row=frame_rows,
            radius=radius
        )

        if graph is not None:
            graphs.append(graph)
        else:
            print("ok")
    return graphs


            

In [12]:
events_model = pd.read_csv("events_model_df.csv")
frames_model = pd.read_csv("frames_model_df.csv")

graphs = build_graph_dataset(events_model,frames_model)
print(len(graphs))
torch.save(graphs, 'worldcup_2022_graphs.pt')

100%|██████████| 187771/187771 [00:10<00:00, 17210.65it/s]


0
